# Translation from English to Spanish using Flan-T5 and Helsinki-NLP/opus-100 Dataset


## Introduction
In this notebook, we will use the Flan-T5 model to perform translation from English to Spanish. We will use the "Helsinki-NLP/opus-100" dataset from Hugging Face, specifically the en-es subset, to train and evaluate our translation model.


## Install Dependencies

In [ ]:
%pip install -r ../requirements.txt -v
# Note: Restart kernel after this installation

## Note Device Configuration

In [ ]:
import torch
import json

USE_CPU = not torch.cuda.is_available()
DEVICE = torch.device("cpu" if USE_CPU else "cuda")

print(f"Using {DEVICE} for execution!")

## Load  Model and Tokenizer

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")


## Loading the Dataset

In [ ]:

from datasets import load_dataset

# Load the Helsinki-NLP/opus-100 dataset
dataset = load_dataset('Helsinki-NLP/opus-100', 'en-es')
print(dataset['train'][0])


## Data Preprocessing

In [ ]:
# Preprocess the dataset for input into the model
def preprocess_data(examples):
    inputs = [f'Translate from English to Spanish: {example["en"]}' for example in examples['translation']]
    targets = [example['es'] for example in examples['translation']]

    # Tokenize inputs
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")

    labels = tokenizer(
        text_target=targets,
        max_length=128,
        truncation=True,
        padding="max_length",
    )
    model_inputs["labels"] = labels["input_ids"]

    # For decoder inputs
    decoder_inputs = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    model_inputs["decoder_input_ids"] = decoder_inputs["input_ids"]

    return model_inputs

# Apply preprocessing to the dataset
train_dataset = dataset['train'].select(range(30000)).map(preprocess_data, batched=True)
test_dataset = dataset['test'].map(preprocess_data, batched=True)

print(train_dataset[0])


## Prepare Datasets for Seq2Seq Training

In [ ]:
def prepare_for_seq2seq_trainer(examples):
    # Replace padding token ids with -100 so that the training loss ignores those positions
    examples["labels"] = [
        [
            -100 if token_id == tokenizer.pad_token_id else token_id
            for token_id in sequence
        ]
        for sequence in examples["labels"]
    ]
    return examples

train_dataset = train_dataset.map(prepare_for_seq2seq_trainer, batched=True)
test_dataset = test_dataset.map(prepare_for_seq2seq_trainer, batched=True)

columns = ["input_ids", "attention_mask", "labels"]
train_dataset = train_dataset.select_columns(columns)
test_dataset = test_dataset.select_columns(columns)

## Freezing the Model

In [ ]:
print(f'Model before freezing LLM layers:\n {model}')

In [ ]:
# Freeze the LLM layer: shared embedding, encoder, and decoder
for module in (model.shared, model.encoder, model.decoder):
    for parameter in module.parameters():
        parameter.requires_grad = False

trainable_params = [name for name, param in model.named_parameters() if param.requires_grad]
print(f' Trainable parameters: {trainable_params}')

In [ ]:
print(f'Model after freezing LLM layers:\n {model}')


### Important Considerations in Transfer Learning

1. **Freezing the LLM Layer:** In transfer learning, it's important to freeze the pre-trained language model layer to retain the knowledge it has already acquired and to avoid overfitting. This allows the model to leverage its pre-trained capabilities while focusing on learning the new task-specific nuances.

2. **Loss Function:** During training, Flan-T5 computes token-level cross-entropy loss from its logits and the provided labels. Padding positions marked with `-100` are ignored. `Seq2SeqTrainer` uses this loss automatically, so no separate `from_logits` setting is needed.


## Model Training

In [ ]:
# Compile the model using Seq2SeqTrainer

from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

if USE_CPU:
    PARAM_EPOCHS = 1
    BATCH_SIZE = 2
    NUM_TRAINING_SAMPLES = 512
    NUM_VALIDATION_SAMPLES = 128
else:
    PARAM_EPOCHS = 3
    BATCH_SIZE = 16
    NUM_TRAINING_SAMPLES = len(train_dataset)
    NUM_VALIDATION_SAMPLES = len(test_dataset)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    return_tensors="pt",
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./flan-t5-english-spanish",
    num_train_epochs=PARAM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    use_cpu=USE_CPU,
    learning_rate=5e-5,
    optim="adamw_torch",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset.select(range(min(NUM_TRAINING_SAMPLES, len(train_dataset)))),
    eval_dataset=test_dataset.select(range(min(NUM_VALIDATION_SAMPLES, len(test_dataset)))),
    data_collator=data_collator,
)

trainer.train()


## Performing Translation

In [ ]:

import torch

model.eval()
DEVICE = next(model.parameters()).device

# Translate and display up to five examples from the test set
for example in test_dataset.select(range(min(5, len(test_dataset)))):
    input_ids = torch.tensor(
        [example["input_ids"]],
        dtype=torch.long,
        device=DEVICE,
    )
    attention_mask = torch.tensor(
        [example["attention_mask"]],
        dtype=torch.long,
        device=DEVICE,
    )

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=128,
            num_beams=4,
        )

    # Replace ignored label positions (-100) with the pad token ID before decoding.
    reference_ids = [
        token_id if token_id != -100 else tokenizer.pad_token_id
        for token_id in example["labels"]
    ]

    print(f'Input:\n {tokenizer.decode(example["input_ids"], skip_special_tokens=True)}')
    print(f'Reference Translation:\n {tokenizer.decode(reference_ids, skip_special_tokens=True)}')
    print(f'Translated Text:\n {tokenizer.decode(output_ids[0], skip_special_tokens=True)}')
    print()

## Calculating ROUGE and BLEU Scores


In [ ]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
nltk.download('punkt')
nltk.download("punkt_tab")


def translate(inputs):
    input_ids = torch.tensor(
        [inputs["input_ids"]],
        dtype=torch.long,
        device=model.device,
    )
    
    attention_mask = torch.tensor(
        [inputs["attention_mask"]],
        dtype=torch.long,
        device=model.device,
    )
    
    outputs = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_length=128,
        num_beams=4,
        early_stopping=True,
    )
        
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# Function to calculate ROUGE and BLEU scores
def calculate_scores(reference, hypothesis):
    # Initialize scorers
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    bleu_smoothing = SmoothingFunction().method4

    # Calculate ROUGE scores
    rouge_scores = rouge.score(reference, hypothesis)

    # Calculate BLEU score
    reference_tokens = [nltk.word_tokenize(reference)]
    hypothesis_tokens = nltk.word_tokenize(hypothesis)
    bleu_score = sentence_bleu(reference_tokens, hypothesis_tokens, smoothing_function=bleu_smoothing)

    return rouge_scores, bleu_score


# Evaluate translations and calculate scores

# Accumulate per-example scores so the dataset can be processed one example at a time.
rouge_totals = {
    metric: {"precision": 0.0, "recall": 0.0, "fmeasure": 0.0}
    for metric in ("rouge1", "rouge2", "rougeL")
}
bleu_total = 0.0
num_examples = 0

if USE_CPU:
    num_to_evaluate = min(5, len(test_dataset))
else:
    num_to_evaluate = len(test_dataset)

dataset_iterator = iter(test_dataset.select(range(num_to_evaluate)))


while (batch := next(dataset_iterator, None)) is not None:
    translated_text = translate(batch)
    
    # undo the padding done earlier #TODO describe better
    label_ids = [
        tokenizer.pad_token_id if token_id == -100 else token_id
        for token_id in batch["labels"]
    ]
    
    reference_text = tokenizer.decode(label_ids, skip_special_tokens=True)
    rouge_scores, bleu_score = calculate_scores(reference_text, translated_text)
    
    for metric, score in rouge_scores.items():
        rouge_totals[metric]["precision"] += score.precision
        rouge_totals[metric]["recall"] += score.recall
        rouge_totals[metric]["fmeasure"] += score.fmeasure

    bleu_total += bleu_score
    num_examples += 1
    
if num_examples == 0:
    raise ValueError("test_dataset is empty; no scores were computed.")

average_rouge_scores = {
    metric: {
        component: total / num_examples
        for component, total in components.items()
    }
    for metric, components in rouge_totals.items()
}
average_bleu = bleu_total / num_examples


In [ ]:
print(f"Examples scored: {num_examples}")
print(f"Average ROUGE scores: {json.dumps(average_rouge_scores, indent=2)}")
print(f"Average sentence BLEU: {average_bleu:.4f}")


## Conclusion
In this notebook, Flan-T5 model is used to perform translation from English to Spanish using the Helsinki-NLP/opus-100 dataset. The following sequence of operations are done in this project to demonstrate the effectiveness of Flan-T5 model for translation:
1. preprocess the dataset
2. freeze the LLM layer
3. fine-tuned the model by training the output layer
4. Performed translations on the fine-tuned model
5. ROUGE and BLEU scores are calculated to assess the quality of the translations. 